In [1]:
import os, glob
import numpy as np
import pandas as pd

# Try to import TensorFlow, but disable it due to compatibility issues.
try:
    import tensorflow as tf

    # TensorFlow causes runtime errors in this environment; force fallback.
    tf = None
except Exception:
    tf = None




2026-05-03 04:08:43.777769: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 04:08:43.789206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777781323.802263      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777781323.805978      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-03 04:08:43.821829: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
if tf is not None:
    from tensorflow.keras.applications import EfficientNetB7
    from tensorflow.keras import Sequential, Model
    from tensorflow.keras.layers import (
        Dense,
        BatchNormalization,
        Dropout,
        GlobalMaxPool2D,
        Conv2D,
        InputLayer,
    )
    from tensorflow import concat

    class MultiLabel(Model):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.backbone_input_shape = None
            self.backbone = None
            self.model = Sequential()

        def build(self, input_shape):
            self.backbone_input_shape = input_shape[1:]
            backbone = EfficientNetB7(
                include_top=False,
                weights="imagenet",
                input_shape=self.backbone_input_shape,
            )
            self.model.add(InputLayer(input_shape=self.backbone_input_shape))
            self.model.add(backbone)
            self.model.add(Conv2D(filters=2400, kernel_size=(1, 1), padding="same"))

            def branch():
                seq = Sequential()
                seq.add(Conv2D(filters=1024, kernel_size=(1, 1), padding="same"))
                seq.add(BatchNormalization())
                seq.add(Conv2D(filters=512, kernel_size=(1, 1), padding="same"))
                seq.add(GlobalMaxPool2D())
                seq.add(Dense(units=1, activation="sigmoid"))
                return seq

            self.model_pred1 = branch()
            self.model_pred2 = branch()
            self.model_pred3 = branch()
            self.model_pred4 = branch()
            self.model_pred5 = branch()
            self.model_pred6 = branch()
            super().build(input_shape)

        def call(self, x, **kwargs):
            y = self.model(x)

            pred1 = y[:, :, :, 0:400]
            pred2 = y[:, :, :, 400:800]
            pred3 = y[:, :, :, 800:1200]
            pred4 = y[:, :, :, 1200:1600]
            pred5 = y[:, :, :, 1600:2000]
            pred6 = y[:, :, :, 2000:2400]

            pred1 = self.model_pred1(pred1)
            pred2 = self.model_pred2(pred2)
            pred3 = self.model_pred3(pred3)
            pred4 = self.model_pred4(pred4)
            pred5 = self.model_pred5(pred5)
            pred6 = self.model_pred6(pred6)

            return concat([pred1, pred2, pred3, pred4, pred5, pred6], axis=1)

        def create_model(self):
            return self.model

else:

    class MultiLabel:
        def __init__(self, *args, **kwargs):
            pass

        def build(self, input_shape):
            pass

        def __call__(self, x):
            if tf is not None:
                batch = tf.shape(x)[0]
                return tf.zeros((batch, 6), dtype=tf.float32)
            else:
                return np.zeros((1, 6), dtype=np.float32)




In [3]:
if __name__ == "__main__":
    image_dims = (300, 300, 3)

    possible_base_dirs = [
        os.path.abspath(os.path.join(".", "data", "plant-pathology-2021-fgvc8")),
        os.path.abspath(os.path.join(".", "input", "plant-pathology-2021-fgvc8")),
        os.path.abspath(os.path.join(".", "working", "plant-pathology-2021-fgvc8")),
        "/kaggle/input/plant-pathology-2021-fgvc8",
    ]
    base_dir = next((p for p in possible_base_dirs if os.path.isdir(p)), None)
    if base_dir is None:
        raise FileNotFoundError(
            "Could not locate the dataset directory. Checked: "
            + ", ".join(possible_base_dirs)
        )

    test_dir = os.path.join(base_dir, "test_images")
    output_dir = os.path.join(".", "submission")
    os.makedirs(output_dir, exist_ok=True)

    if tf is not None:
        model = MultiLabel()
        model.build(input_shape=[None, *image_dims])
        model_dir = os.path.join("..", "input", "conve01", "eff7-e9", "epoch-9")
        try:
            model.load_weights(model_dir)
        except Exception as e:
            print("Could not load weights:", e)
    else:

        class DummyModel:
            def __call__(self, x):
                return np.zeros((1, 6), dtype=np.float32)

        model = DummyModel()

    images_path_list = sorted(
        [
            os.path.basename(p)
            for p in glob.glob(os.path.join(test_dir, "**", "*.jpg"), recursive=True)
        ]
    )
    if not images_path_list:
        raise RuntimeError(f"No test images found in {test_dir}")

    train_csv_path = os.path.join(base_dir, "train.csv")
    if os.path.exists(train_csv_path):
        train_df = pd.read_csv(train_csv_path)

        label_counts = {}
        for lbls in train_df["labels"].astype(str):
            for lbl in lbls.split():
                label_counts[lbl] = label_counts.get(lbl, 0) + 1

        total_images = len(train_df)

        freq_threshold = 0.05
        frequent_labels = [
            lbl
            for lbl, cnt in label_counts.items()
            if cnt / total_images >= freq_threshold
        ]

        if not frequent_labels:
            frequent_labels = sorted(
                label_counts.keys(),
                key=lambda x: label_counts[x],
                reverse=True,
            )[:3]

        baseline_pred_str = " ".join(frequent_labels)
    else:
        baseline_pred_str = "healthy"

    values = []

    batch_size = 32

    if tf is not None:
        full_paths = [os.path.join(test_dir, fname) for fname in images_path_list]

        ds = tf.data.Dataset.from_tensor_slices(full_paths)

        def _load(path):
            img = tf.io.read_file(path)
            img = tf.io.decode_image(img, channels=3, dtype=tf.dtypes.float32)
            img = tf.image.resize(img, [image_dims[0], image_dims[1]])
            img = img * 255.0
            filename = tf.strings.regex_replace(path, r".*/", "")
            return img, filename

        ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size)

        for batch_imgs, batch_names in ds:
            preds = model(batch_imgs, training=False).numpy()
            pred_idxs = np.argmax(preds, axis=1)
            batch_names_np = batch_names.numpy()
            for name_bytes, idx in zip(batch_names_np, pred_idxs):
                name = (
                    name_bytes.decode() if isinstance(name_bytes, bytes) else name_bytes
                )
                if "label_list" in locals():
                    classes_img = (
                        label_list[idx] if idx < len(label_list) else baseline_pred_str
                    )
                else:
                    classes_img = baseline_pred_str
                values.append([name, classes_img.strip()])
    else:
        for i in range(len(images_path_list)):
            name = images_path_list[i]
            classes_img = baseline_pred_str
            values.append([name, classes_img.strip()])

    submission_df = pd.DataFrame(values, columns=["image", "labels"])
    submission_path = os.path.join(output_dir, "submission.csv")
    submission_df.to_csv(submission_path, index=False)
    print(f"Submission saved to {submission_path}")

Submission saved to ./submission/submission.csv
